# TensorRT-LLM

**Stage 5: Inference Optimization - Notebook 58**

This notebook explores TensorRT-LLM, NVIDIA's optimized inference framework for LLMs. We'll cover:
- TensorRT-LLM overview and architecture
- NVIDIA-specific optimizations
- Model conversion workflow
- Quantization with TRT-LLM
- Inference benchmarks (vs PyTorch, vs vLLM)
- When to use TensorRT-LLM
- Deployment guide and best practices

In [ ]:
# Note: This notebook demonstrates TensorRT-LLM concepts
# Actual usage requires NVIDIA GPU and TensorRT-LLM installation

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional
import json

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

## 1. TensorRT-LLM Overview

### Historical Context

**Background**:
- TensorRT: NVIDIA's inference optimizer (since 2016)
- Originally for CNNs, later expanded to transformers
- TensorRT-LLM: Specialized for large language models (2023)

**Evolution**:
```
2016: TensorRT 1.0 - CNN optimization
2019: TensorRT 6.0 - Transformer support
2021: FasterTransformer - NVIDIA's LLM library
2023: TensorRT-LLM - Unified LLM inference (replaces FasterTransformer)
2024: TensorRT-LLM v0.8+ - Production ready, vLLM integration
```

**Key Innovations**:
1. **Hardware-Aware Compilation**: Optimizes for specific GPU architecture
2. **Kernel Fusion**: Combines operations to reduce memory transfers
3. **Mixed Precision**: Automatic FP16/INT8 optimization
4. **PagedAttention**: Adopted from vLLM
5. **In-Flight Batching**: Dynamic batching like vLLM
6. **Multi-GPU**: Optimized tensor and pipeline parallelism

**Architecture**:
```
┌─────────────────────────────────────────────┐
│        TensorRT-LLM Architecture            │
├─────────────────────────────────────────────┤
│                                             │
│  ┌──────────────────────────────────────┐  │
│  │  Model Definition (Python)           │  │
│  │  - HuggingFace → TRT-LLM            │  │
│  │  - Custom architectures              │  │
│  └──────────────┬───────────────────────┘  │
│                 │                          │
│                 ▼                          │
│  ┌──────────────────────────────────────┐  │
│  │  Build Phase                         │  │
│  │  - Graph optimization                │  │
│  │  - Kernel fusion                     │  │
│  │  - Quantization (FP16/INT8/INT4)    │  │
│  └──────────────┬───────────────────────┘  │
│                 │                          │
│                 ▼                          │
│  ┌──────────────────────────────────────┐  │
│  │  TensorRT Engine (.engine file)     │  │
│  │  - GPU-specific binary               │  │
│  │  - Optimized for target hardware     │  │
│  └──────────────┬───────────────────────┘  │
│                 │                          │
│                 ▼                          │
│  ┌──────────────────────────────────────┐  │
│  │  Runtime Execution                   │  │
│  │  - Custom CUDA kernels               │  │
│  │  - PagedAttention                    │  │
│  │  - In-flight batching                │  │
│  └──────────────────────────────────────┘  │
│                                             │
└─────────────────────────────────────────────┘
```

**Key Difference from vLLM**:
- vLLM: Runtime optimization (works with any PyTorch model)
- TensorRT-LLM: Compile-time optimization (requires model conversion)
- Trade-off: More setup but potentially higher performance

In [ ]:
# Visualize optimization pipeline
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Stages
stages = [
    (1, 8, "PyTorch Model\n(Hugging Face)", '#3498db'),
    (1, 6, "TRT-LLM\nModel Definition", '#9b59b6'),
    (1, 4, "Build Engine\n(Compile)", '#e67e22'),
    (1, 2, "TensorRT Engine\n(.engine)", '#e74c3c'),
    (1, 0.5, "Optimized\nInference", '#2ecc71'),
]

for x, y, label, color in stages:
    rect = plt.Rectangle((x, y-0.4), 2, 0.8, color=color, alpha=0.7, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 1, y, label, ha='center', va='center', fontsize=10, fontweight='bold')
    
    if y > 1:
        ax.arrow(2, y-0.5, 0, -0.9, head_width=0.15, head_length=0.1, fc='black', ec='black')

# Optimizations at each stage
optimizations = [
    (4.5, 7.5, "Conversions:\n• Weights\n• Architecture", '#95a5a6'),
    (4.5, 5.5, "Graph Opts:\n• Kernel fusion\n• Memory layout\n• Quantization", '#95a5a6'),
    (4.5, 3.5, "Hardware Opts:\n• GPU-specific\n• Tensor cores\n• Memory hierarchy", '#95a5a6'),
    (4.5, 1.5, "Runtime Opts:\n• PagedAttention\n• In-flight batching\n• Streaming", '#95a5a6'),
]

for x, y, text, color in optimizations:
    bbox = dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3, edgecolor='black')
    ax.text(x, y, text, ha='left', va='center', fontsize=9, bbox=bbox)

ax.set_title('TensorRT-LLM Optimization Pipeline', fontsize=14, fontweight='bold', pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

print("Key Characteristics:")
print("1. Compile-time optimization: One-time cost for maximum performance")
print("2. Hardware-specific: Engine optimized for target GPU")
print("3. Static graph: Model architecture fixed at build time")
print("4. Quantization: INT8/INT4 for extreme performance")

## 2. NVIDIA-Specific Optimizations

TensorRT-LLM leverages NVIDIA hardware features not available to generic frameworks.

In [ ]:
print("NVIDIA Hardware Optimizations")
print("="*80)

hardware_opts = """
1. TENSOR CORES:

   What: Specialized hardware for matrix multiplication
   
   Generations:
   - Volta (V100):     Mixed precision (FP16)
   - Turing (T4):      INT8, INT4
   - Ampere (A100):    TF32, BF16, FP8 (sparse)
   - Hopper (H100):    FP8, FP4, Transformer Engine
   
   Performance (vs CUDA cores):
   - FP16: 8-16x faster
   - INT8: 16-32x faster
   - INT4: 32-64x faster
   
   TensorRT-LLM automatically uses Tensor Cores when beneficial.


2. FLASH ATTENTION 2:

   - Fused attention kernel (fewer memory transfers)
   - Optimized for Ampere/Hopper
   - 2-4x faster than standard attention
   - Integrated by default in TensorRT-LLM


3. CUDA GRAPHS:

   - Capture entire execution graph
   - Eliminate kernel launch overhead
   - Especially beneficial for small batches
   - 10-30% speedup for latency-critical workloads


4. TRANSFORMER ENGINE (H100):

   - Automatic FP8 quantization
   - Dynamic range scaling
   - 2x speedup vs FP16 with minimal quality loss
   - Only on Hopper GPUs


5. MULTI-GPU OPTIMIZATIONS:

   NVLink:
   - High-speed GPU interconnect (600-900 GB/s per GPU)
   - Much faster than PCIe (32 GB/s)
   - Critical for tensor parallelism
   
   NCCL:
   - Optimized collective operations
   - All-reduce, all-gather for model parallelism
   - Topology-aware routing


6. MEMORY HIERARCHY:

   L1/L2 Cache:
   - Automatic caching of frequently accessed data
   - Shared memory optimizations
   
   Unified Memory:
   - Automatic CPU-GPU memory management
   - Useful for very large models


7. KERNEL FUSION:

   Common patterns fused:
   - LayerNorm + Linear
   - GELU activation + Linear
   - Attention + Residual + LayerNorm
   
   Benefits:
   - Fewer memory reads/writes
   - Reduced kernel launch overhead
   - Better cache utilization
   - 20-40% speedup
"""

print(hardware_opts)

# Compare GPU capabilities
gpu_comparison = {
    'GPU': ['V100', 'T4', 'A10', 'A100', 'H100'],
    'Memory (GB)': [16, 16, 24, 80, 80],
    'FP16 TFLOPS': [125, 65, 125, 312, 756],
    'INT8 TOPS': [None, 260, 500, 1248, 3026],
    'Memory BW (GB/s)': [900, 320, 600, 2000, 3350],
    'Best For': [
        'Legacy/Small models',
        'Edge inference',
        'Balanced',
        'Large models',
        'Extreme performance'
    ]
}

import pandas as pd
df = pd.DataFrame(gpu_comparison)
print("\n\nGPU Comparison:")
print("="*80)
print(df.to_string(index=False))

## 3. Model Conversion Workflow

In [ ]:
print("TensorRT-LLM Model Conversion Guide")
print("="*80)

conversion_guide = """
STEP-BY-STEP CONVERSION:

1. INSTALLATION:

   # Install TensorRT-LLM
   pip install tensorrt_llm==0.8.0
   
   # Or use Docker (recommended)
   docker pull nvcr.io/nvidia/tensorrt_llm/release:latest


2. CONVERT HUGGING FACE MODEL:

   # Example: Llama 2 7B
   python examples/llama/convert_checkpoint.py \
       --model_dir meta-llama/Llama-2-7b-hf \
       --output_dir ./llama-2-7b-trt \
       --dtype float16
   
   # This creates:
   # - config.json: Model configuration
   # - rank*.safetensors: Converted weights


3. BUILD TENSORRT ENGINE:

   # Single GPU
   trtllm-build \
       --checkpoint_dir ./llama-2-7b-trt \
       --output_dir ./llama-2-7b-engine \
       --gemm_plugin float16 \
       --max_batch_size 64 \
       --max_input_len 2048 \
       --max_output_len 512
   
   # Multi-GPU (tensor parallelism)
   trtllm-build \
       --checkpoint_dir ./llama-2-70b-trt \
       --output_dir ./llama-2-70b-engine \
       --gemm_plugin float16 \
       --gpt_attention_plugin float16 \
       --tp_size 4 \
       --workers 4 \
       --max_batch_size 128 \
       --max_input_len 4096 \
       --max_output_len 1024
   
   Build time: 5-30 minutes (one-time cost)


4. RUN INFERENCE:

   # Python API
   from tensorrt_llm import LLM
   
   llm = LLM(
       model_dir="./llama-2-7b-engine",
       tokenizer_dir="meta-llama/Llama-2-7b-hf"
   )
   
   outputs = llm.generate(
       prompts=["Hello, how are you?"],
       max_new_tokens=100,
       temperature=0.7,
   )
   
   # Triton Inference Server (production)
   # See deployment section below


5. SUPPORTED MODELS (as of v0.8):

   ✓ GPT (GPT-2, GPT-J, GPT-NeoX)
   ✓ Llama (Llama 1/2, CodeLlama)
   ✓ Mistral (Mistral 7B, Mixtral 8x7B)
   ✓ Falcon
   ✓ MPT
   ✓ BLOOM
   ✓ ChatGLM
   ✓ Qwen
   ✓ Baichuan
   
   Check examples/ directory for specific model scripts.


6. QUANTIZATION DURING BUILD:

   # INT8 Weight-Only
   trtllm-build \
       --checkpoint_dir ./model \
       --output_dir ./engine \
       --use_weight_only \
       --weight_only_precision int8
   
   # INT4 Weight-Only (AWQ)
   trtllm-build \
       --checkpoint_dir ./model \
       --output_dir ./engine \
       --use_weight_only \
       --weight_only_precision int4_awq \
       --per_group
   
   # SmoothQuant (INT8 activations + weights)
   trtllm-build \
       --checkpoint_dir ./model \
       --output_dir ./engine \
       --use_smooth_quant \
       --per_token \
       --per_channel


7. ADVANCED OPTIONS:

   PagedAttention:
   --paged_kv_cache --remove_input_padding
   
   Multi-Query Attention:
   --multi_query_mode (for models like Falcon)
   
   In-Flight Batching:
   --max_num_tokens 8192 (total tokens in flight)
   
   Custom Plugins:
   --use_custom_all_reduce (for multi-node)
"""

print(conversion_guide)

## 4. Quantization with TensorRT-LLM

TensorRT-LLM supports various quantization schemes for extreme performance.

In [ ]:
# Compare quantization schemes
quantization_comparison = {
    'Method': [
        'FP16 (baseline)',
        'INT8 Weight-Only',
        'INT4 Weight-Only (GPTQ)',
        'INT4 Weight-Only (AWQ)',
        'SmoothQuant INT8',
        'FP8 (H100 only)',
    ],
    'Memory Reduction': ['1x', '2x', '4x', '4x', '2-3x', '2x'],
    'Speedup': ['1x', '1.5-2x', '3-4x', '3-4x', '2-3x', '2x'],
    'Quality Impact': ['None', 'Minimal', 'Small', 'Small', 'Minimal', 'Minimal'],
    'Calibration': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No'],
    'Best For': [
        'Quality critical',
        'Quick start',
        'Extreme compression',
        'Best quality @ INT4',
        'Activation compression',
        'H100 max performance',
    ]
}

df = pd.DataFrame(quantization_comparison)
print("Quantization Methods Comparison")
print("="*90)
print(df.to_string(index=False))

# Visualize trade-offs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = ['FP16', 'INT8-WO', 'INT4-GPTQ', 'INT4-AWQ', 'SmoothQuant', 'FP8']
memory = [1.0, 0.5, 0.25, 0.25, 0.35, 0.5]
speedup = [1.0, 1.75, 3.5, 3.5, 2.5, 2.0]
quality = [100, 98, 94, 96, 97, 98]  # Relative quality

# Memory vs Speedup
ax = axes[0]
colors_methods = ['#e74c3c', '#e67e22', '#f39c12', '#f1c40f', '#2ecc71', '#3498db']
scatter = ax.scatter(memory, speedup, s=[q*5 for q in quality], 
                    c=colors_methods, alpha=0.6, edgecolors='black', linewidth=2)

for i, method in enumerate(methods):
    ax.annotate(method, (memory[i], speedup[i]), 
               xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')

ax.set_xlabel('Relative Memory Usage', fontsize=12)
ax.set_ylabel('Speedup vs FP16', fontsize=12)
ax.set_title('Memory vs Speed Trade-off', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.invert_xaxis()  # Lower memory is better

# Add legend for quality (bubble size)
for q_val, q_label in [(100, '100%'), (95, '95%'), (90, '90%')]:
    ax.scatter([], [], s=q_val*5, c='gray', alpha=0.5, label=f'{q_label} quality')
ax.legend(scatterpoints=1, frameon=True, labelspacing=2, title='Quality')

# Performance bars
ax = axes[1]
x = np.arange(len(methods))
width = 0.35

bars1 = ax.bar(x - width/2, speedup, width, label='Speedup', 
               color='#2ecc71', alpha=0.7)
bars2 = ax.bar(x + width/2, quality, width, label='Quality (%)',
               color='#3498db', alpha=0.7)

ax.set_xlabel('Method', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('Speedup vs Quality', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n\nQuantization Guidelines:")
print("="*80)
print("""
1. STARTING POINT: INT8 Weight-Only
   - No calibration needed
   - 2x memory reduction
   - 1.5-2x speedup
   - Minimal quality loss

2. MAXIMUM COMPRESSION: INT4-AWQ
   - 4x memory reduction (run 70B on single A100!)
   - 3-4x speedup
   - Requires calibration data
   - Best INT4 quality

3. ACTIVATION COMPRESSION: SmoothQuant
   - Quantizes both weights and activations
   - Better for memory-bandwidth bound workloads
   - Requires calibration

4. H100 ONLY: FP8
   - Hardware-accelerated
   - Best speedup with minimal quality loss
   - Only works on Hopper GPUs
""")

## 5. Inference Benchmarks

In [ ]:
# Benchmark data (based on NVIDIA's published results)
# Llama 2 7B on A100 80GB

benchmark_results = {
    'Framework': [
        'PyTorch (HF)',
        'PyTorch (HF) + torch.compile',
        'vLLM',
        'TRT-LLM FP16',
        'TRT-LLM INT8-WO',
        'TRT-LLM INT4-AWQ',
    ],
    'Throughput (tokens/s)': [150, 280, 1200, 1850, 2400, 3500],
    'Latency P50 (ms)': [450, 320, 120, 85, 65, 45],
    'Memory (GB)': [14, 14, 8.5, 8.2, 5.8, 4.2],
    'First Token (ms)': [180, 180, 45, 30, 25, 20],
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

frameworks = benchmark_results['Framework']
colors_bench = ['#e74c3c', '#e67e22', '#f39c12', '#2ecc71', '#27ae60', '#16a085']

# Throughput
ax = axes[0, 0]
bars = ax.barh(frameworks, benchmark_results['Throughput (tokens/s)'],
               color=colors_bench, alpha=0.7)
ax.set_xlabel('Throughput (tokens/s)', fontsize=12)
ax.set_title('Throughput Comparison (Llama 2 7B, A100)', 
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for i, bar in enumerate(bars):
    width = bar.get_width()
    speedup = width / benchmark_results['Throughput (tokens/s)'][0]
    ax.text(width, bar.get_y() + bar.get_height()/2,
            f' {int(width)} ({speedup:.1f}x)',
            ha='left', va='center', fontsize=9, fontweight='bold')

# Latency
ax = axes[0, 1]
bars = ax.barh(frameworks, benchmark_results['Latency P50 (ms)'],
               color=colors_bench, alpha=0.7)
ax.set_xlabel('Latency P50 (ms)', fontsize=12)
ax.set_title('Latency Comparison (lower is better)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_xaxis()

# Memory
ax = axes[1, 0]
bars = ax.barh(frameworks, benchmark_results['Memory (GB)'],
               color=colors_bench, alpha=0.7)
ax.set_xlabel('Memory Usage (GB)', fontsize=12)
ax.set_title('Memory Usage (lower is better)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_xaxis()

for i, bar in enumerate(bars):
    width = bar.get_width()
    reduction = benchmark_results['Memory (GB)'][0] / width
    ax.text(width, bar.get_y() + bar.get_height()/2,
            f' {width:.1f} GB ({reduction:.1f}x)',
            ha='left', va='center', fontsize=9, fontweight='bold')

# Time to first token
ax = axes[1, 1]
bars = ax.barh(frameworks, benchmark_results['First Token (ms)'],
               color=colors_bench, alpha=0.7)
ax.set_xlabel('Time to First Token (ms)', fontsize=12)
ax.set_title('Latency to First Token (lower is better)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_xaxis()

plt.tight_layout()
plt.show()

print("\nKey Findings:")
print("="*80)
print(f"\n1. Throughput:")
print(f"   TRT-LLM INT4-AWQ: {benchmark_results['Throughput (tokens/s)'][-1] / benchmark_results['Throughput (tokens/s)'][0]:.1f}x vs PyTorch")
print(f"   TRT-LLM INT4-AWQ: {benchmark_results['Throughput (tokens/s)'][-1] / benchmark_results['Throughput (tokens/s)'][2]:.1f}x vs vLLM")

print(f"\n2. Memory:")
print(f"   TRT-LLM INT4-AWQ: {benchmark_results['Memory (GB)'][0] / benchmark_results['Memory (GB)'][-1]:.1f}x reduction vs PyTorch")
print(f"   Can run 70B model on single A100 with INT4!")

print(f"\n3. Latency:")
print(f"   TRT-LLM INT4-AWQ: {benchmark_results['Latency P50 (ms)'][0] / benchmark_results['Latency P50 (ms)'][-1]:.1f}x faster vs PyTorch")
print(f"   First token: {benchmark_results['First Token (ms)'][0] / benchmark_results['First Token (ms)'][-1]:.1f}x faster")

print(f"\n4. When TRT-LLM Wins:")
print(f"   - Small batch sizes (latency critical)")
print(f"   - Need extreme quantization (INT4)")
print(f"   - NVIDIA GPUs only")
print(f"   - Can afford build time")

## 6. When to Use TensorRT-LLM

In [ ]:
# Decision matrix
decision_scenarios = [
    {
        'Scenario': 'Low-latency serving (P50 < 100ms)',
        'TensorRT-LLM': 'YES',
        'vLLM': 'MAYBE',
        'PyTorch': 'NO',
        'Reasoning': 'TRT-LLM has lowest latency, especially with quantization',
    },
    {
        'Scenario': 'High-throughput serving (>1000 tokens/s)',
        'TensorRT-LLM': 'YES',
        'vLLM': 'YES',
        'PyTorch': 'NO',
        'Reasoning': 'Both TRT-LLM and vLLM excel, TRT-LLM slightly faster',
    },
    {
        'Scenario': 'Memory-constrained (need INT4)',
        'TensorRT-LLM': 'YES',
        'vLLM': 'NO',
        'PyTorch': 'NO',
        'Reasoning': 'TRT-LLM has best INT4 support (AWQ, GPTQ)',
    },
    {
        'Scenario': 'Multi-GPU (4-8 GPUs)',
        'TensorRT-LLM': 'YES',
        'vLLM': 'YES',
        'PyTorch': 'MAYBE',
        'Reasoning': 'Both optimized for multi-GPU, TRT-LLM better for Hopper',
    },
    {
        'Scenario': 'Rapid prototyping/Research',
        'TensorRT-LLM': 'NO',
        'vLLM': 'YES',
        'PyTorch': 'YES',
        'Reasoning': 'TRT-LLM build time too slow for iteration',
    },
    {
        'Scenario': 'AMD/Intel GPUs',
        'TensorRT-LLM': 'NO',
        'vLLM': 'MAYBE',
        'PyTorch': 'YES',
        'Reasoning': 'TRT-LLM is NVIDIA-only',
    },
    {
        'Scenario': 'Custom model architecture',
        'TensorRT-LLM': 'MAYBE',
        'vLLM': 'MAYBE',
        'PyTorch': 'YES',
        'Reasoning': 'May require custom plugins, more work',
    },
    {
        'Scenario': 'Production deployment (H100)',
        'TensorRT-LLM': 'YES',
        'vLLM': 'YES',
        'PyTorch': 'NO',
        'Reasoning': 'TRT-LLM best for Hopper (FP8), vLLM more flexible',
    },
]

df = pd.DataFrame(decision_scenarios)
print("\nTensorRT-LLM vs Alternatives - Decision Matrix")
print("="*100)
print(df.to_string(index=False))

print("\n\nQuick Decision Guide:")
print("="*80)
print("""
USE TensorRT-LLM when:
✓ Need absolute minimum latency
✓ NVIDIA GPUs (especially Hopper H100)
✓ Can afford one-time build cost
✓ Model is in supported list
✓ Need INT4/INT8 quantization
✓ Production deployment with stable model

USE vLLM when:
✓ Need high throughput
✓ Rapid iteration important
✓ Custom model architectures
✓ Good balance of features
✓ Don't need extreme quantization
✓ Prefer Python-native workflow

USE PyTorch when:
✓ Research/prototyping
✓ Custom architectures
✓ Non-NVIDIA hardware
✓ Learning/debugging
✓ Throughput not critical

HYBRID APPROACH:
- Develop with PyTorch
- Test with vLLM
- Deploy with TensorRT-LLM (if NVIDIA)
""")

## 7. Production Deployment

In [ ]:
print("TensorRT-LLM Production Deployment Guide")
print("="*80)

deployment_guide = """
1. TRITON INFERENCE SERVER (Recommended):

   Why Triton:
   - NVIDIA's production serving platform
   - Native TensorRT-LLM support
   - Advanced features: dynamic batching, model ensemble, metrics
   - Industry standard for NVIDIA deployments
   
   Setup:
   # Directory structure
   model_repository/
   └── llama-2-7b/
       ├── config.pbtxt
       └── 1/
           └── model.plan  # TensorRT engine
   
   # config.pbtxt
   name: "llama-2-7b"
   backend: "tensorrtllm"
   max_batch_size: 128
   
   dynamic_batching {
     max_queue_delay_microseconds: 100
   }
   
   # Start server
   docker run --gpus all --rm \
     -p 8000:8000 -p 8001:8001 -p 8002:8002 \
     -v /path/to/model_repository:/models \
     nvcr.io/nvidia/tritonserver:23.12-trtllm-python-py3 \
     tritonserver --model-repository=/models


2. DOCKER DEPLOYMENT:

   # Dockerfile
   FROM nvcr.io/nvidia/tensorrt_llm/release:latest
   
   WORKDIR /app
   COPY ./engines /app/engines
   COPY ./server.py /app/
   
   EXPOSE 8000
   CMD ["python", "server.py"]
   
   # Build and run
   docker build -t my-llm-server .
   docker run --gpus all -p 8000:8000 my-llm-server


3. KUBERNETES DEPLOYMENT:

   # deployment.yaml
   apiVersion: apps/v1
   kind: Deployment
   metadata:
     name: tensorrt-llm
   spec:
     replicas: 2
     template:
       spec:
         containers:
         - name: tensorrt-llm
           image: my-llm-server:latest
           resources:
             limits:
               nvidia.com/gpu: 1
           ports:
           - containerPort: 8000
         nodeSelector:
           accelerator: nvidia-a100


4. LOAD BALANCING:

   Options:
   a) Multiple instances + load balancer
   b) KServe for Kubernetes
   c) NVIDIA Inference Server (NIS) for edge
   
   Considerations:
   - Each instance loads full model (memory overhead)
   - GPU affinity important
   - Sticky sessions for streaming


5. MONITORING:

   Triton Metrics (Prometheus):
   nv_inference_request_success
   nv_inference_request_failure
   nv_inference_queue_duration_us
   nv_inference_compute_duration_us
   nv_gpu_utilization
   nv_gpu_memory_used_bytes
   
   Custom Metrics:
   from prometheus_client import Counter, Histogram
   
   request_counter = Counter(
       'llm_requests_total',
       'Total LLM requests'
   )
   
   token_histogram = Histogram(
       'llm_tokens_generated',
       'Tokens generated per request'
   )


6. OPTIMIZATION CHECKLIST:

   Build-time:
   ✓ Use appropriate quantization (INT8/INT4)
   ✓ Set max_batch_size to expected peak
   ✓ Tune max_input_len and max_output_len
   ✓ Enable plugins (gemm, gpt_attention)
   ✓ Build for specific GPU architecture
   
   Runtime:
   ✓ Warm up model before serving traffic
   ✓ Monitor GPU utilization (>85%)
   ✓ Use dynamic batching
   ✓ Profile different batch sizes
   ✓ Set appropriate timeouts


7. COMMON PITFALLS:

   ✗ Building on different GPU than deployment
   ✗ Not warming up model (first request slow)
   ✗ Setting max lengths too large (wastes memory)
   ✗ Not monitoring queue lengths
   ✗ Using wrong CUDA/cuDNN versions
   ✗ Ignoring calibration for INT8/INT4
   ✗ Not testing different quantization options


8. PRODUCTION CHECKLIST:

   Before deployment:
   ✓ Benchmark on production hardware
   ✓ Test with production traffic patterns
   ✓ Validate output quality
   ✓ Load test (stress test)
   ✓ Set up monitoring and alerting
   ✓ Document build configuration
   ✓ Plan rollback strategy
   ✓ Test disaster recovery
"""

print(deployment_guide)

## Summary

### Key Takeaways

1. **TensorRT-LLM Architecture**:
   - Compile-time optimization (build once, run fast)
   - Hardware-specific (optimized for target GPU)
   - Kernel fusion and graph optimization
   - Native support for NVIDIA features

2. **Performance Advantages**:
   - **Latency**: 3-5x faster than PyTorch, 1.5-2x faster than vLLM
   - **Throughput**: Up to 23x vs PyTorch, 2-3x vs vLLM
   - **Memory**: INT4 enables 4x compression (70B on single A100!)
   - **First token**: Lowest latency of any framework

3. **Quantization**:
   - **INT8**: 2x memory, 1.5-2x speed, minimal quality loss
   - **INT4-AWQ**: 4x memory, 3-4x speed, best INT4 quality
   - **SmoothQuant**: Quantize activations too
   - **FP8**: H100-only, 2x speed with near-FP16 quality

4. **When to Use**:
   - ✓ Production deployment on NVIDIA GPUs
   - ✓ Latency-critical applications
   - ✓ Need extreme compression (INT4)
   - ✓ Stable model (can afford build time)
   - ✓ Hopper GPUs (H100) for maximum benefit

5. **When NOT to Use**:
   - ✗ Rapid prototyping (build time 5-30 min)
   - ✗ Non-NVIDIA hardware
   - ✗ Frequently changing models
   - ✗ Custom architectures (may need plugins)
   - ✗ Limited engineering resources

6. **Best Practices**:
   - Start with INT8 weight-only quantization
   - Use Triton Inference Server for production
   - Build on same GPU as deployment
   - Calibrate for INT4/INT8 with representative data
   - Monitor GPU utilization (should be >85%)
   - Load test before production

### TensorRT-LLM vs vLLM

| Aspect | TensorRT-LLM | vLLM |
|--------|--------------|------|
| **Latency** | Best (especially INT4) | Good |
| **Throughput** | Best (with quantization) | Excellent |
| **Setup** | Complex (build required) | Simple |
| **Flexibility** | Low (static graph) | High |
| **Hardware** | NVIDIA only | Any GPU |
| **Quantization** | INT4/INT8/FP8 | Limited |
| **Memory** | Best (with INT4) | Good |
| **Iteration** | Slow (rebuild needed) | Fast |

### Recommendation

**For most users**: Start with vLLM
- Easier setup
- Faster iteration
- Good enough performance

**Migrate to TensorRT-LLM when**:
- Need absolute minimum latency
- Have NVIDIA GPUs (especially H100)
- Model is stable (no frequent changes)
- Can justify engineering effort
- Need INT4 quantization

**Hybrid approach**:
- Develop with PyTorch
- Test with vLLM  
- Deploy with TensorRT-LLM (if justified)

### Next Steps

- **Notebook 59**: Continuous Batching (deep dive into scheduling)